# mantiq: answers from your documents, with sources and a confidence check

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ahmed-Ousama/mantiq/blob/main/examples/mantiq-demo.ipynb)

[mantiq](https://github.com/Ahmed-Ousama/mantiq) reads your documents into the model's KV cache once, then answers any number of questions from them, without a retrieval pipeline. This notebook walks through the library with Qwen3-4B: answers, citations, answer checks, conversations and live updates. It takes about 10 minutes, most of it downloading the model.

**Before you start:** Runtime → Change runtime type → T4 GPU. On a GPU, the model loads in 4-bit automatically.

[GitHub](https://github.com/Ahmed-Ousama/mantiq) · [Paper (PDF)](https://github.com/Ahmed-Ousama/mantiq/releases/download/v0.2.0/mantiq-paper.pdf)

In [1]:
!pip install -q mantiq

## Load the model

Qwen3 thinks before it answers by default; with thinking off, answers are short and fast. mantiq has built-in citation heads and an answer check for this model.

In [2]:
from mantiq import NaiveCAG

cag = NaiveCAG("Qwen/Qwen3-4B", chat_template_kwargs={"enable_thinking": False})

## Read the knowledge once

The knowledge is a few documents from an electronics store. mantiq runs the model over them once and keeps the KV cache. The document that changes most often, this week's offer, comes last, so updating it later is cheap.

In [3]:
documents = {
    "tv.md": "The Vision 55 is a 55-inch 4K TV with HDR10 and smart apps. It costs $499 and comes with a 2-year warranty.",
    "fridge.md": "The FrostFree is a 450-liter refrigerator with an A++ energy rating. It costs $899, with a 5-year warranty on the compressor.",
    "washer.md": "The CleanWave is a 9 kg front-load washing machine whose wash cycles adapt to the load. It costs $649.",
    "delivery.md": "Delivery takes 2 to 4 business days. It is free for orders over $300, otherwise $25.",
    "returns.md": "Returns are accepted within 14 days with the receipt. Refunds go to the original payment method.",
    "offers.md": "This week: 10% off the CleanWave washer.",
}

kv_cache, read_seconds = cag.prepare_kvcache_from_loader(
    documents,
    prompt_instruction="You are a sales agent for an electronics store.",
    answer_instruction="Answer in one or two sentences, using only the given context.",
)
print(f"Read {kv_cache.get_seq_length()} tokens of knowledge in {read_seconds:.1f} s, once.")

Read 199 tokens of knowledge in 1.0 s, once.


## Ask questions

Each question reads only its own few tokens on top of the cache, so the answer starts right away.

In [4]:
import time

for question in ["I want a 4K screen, what do you have?", "How long does delivery take?", "Is there a deal on washing machines?"]:
    start = time.perf_counter()
    print(f"Q: {question}\nA: ", end="")
    for chunk in cag.stream_response(question, kv_cache, max_new_tokens=80):
        print(chunk, end="", flush=True)
    print(f"\n   ({time.perf_counter() - start:.1f} s)\n")

Q: I want a 4K screen, what do you have?
A: We have the Vision 55, a 55-inch 4K TV with HDR10 and smart apps, priced at $499 with a 2-year warranty.
   (4.5 s)

Q: How long does delivery take?
A: Delivery takes 2 to 4 business days.
   (1.3 s)

Q: Is there a deal on washing machines?
A: Yes, there is a 10% off deal on the CleanWave washing machine this week.
   (2.4 s)



## Where does an answer come from?

`cite()` finds the document an answer draws on, in one pass: a few attention heads in the model look at the source, and mantiq knows which ones for 11 models. `supported` is low when the answer doesn't come from the knowledge at all, as for the last question.

In [5]:
for question in ["Can I send back a TV I don't like?", "Which fridge do you have, and is it efficient?", "Do you sell gaming consoles?"]:
    answer = cag.generate_response(question, kv_cache, max_new_tokens=60).strip()
    cited = cag.cite(question, answer, kv_cache, top_k=1)
    print(f"Q: {question}\nA: {answer}\n   source: {cited.citations[0].document}   supported: {cited.supported:.2f}\n")

Q: Can I send back a TV I don't like?
A: Yes, you can return the TV within 14 days of purchase with a receipt.
   source: returns.md   supported: 1.00

Q: Which fridge do you have, and is it efficient?
A: I have the FrostFree refrigerator, which is efficient due to its A++ energy rating.
   source: fridge.md   supported: 1.00

Q: Do you sell gaming consoles?
A: No, we don't sell gaming consoles, but we do sell the Vision 55 TV, which is great for gaming with its 4K resolution and HDR10 support.
   source: tv.md   supported: 0.02



## Is the answer right?

Most wrong answers are not made up from nothing: the model takes a fact from the wrong document, or slips on a step. `confidence` is meant for those. A small check reads the model's internal state as it answers; it was trained without reference answers, to predict whether the model's own thinking would agree. `generate_checked_response()` answers fast, and thinks it over only when the confidence is below 0.75. The check isn't perfect: it also sends some right answers to thinking, which only costs time.

In [6]:
questions = [
    "How long is the TV's warranty?",
    "Which product has the longest warranty?",
    "Does the washer come with a warranty?",
    "Which is cheaper this week, the washer or the TV?",
    "What do the TV and the washer cost together this week, with delivery?",
    "If I only buy a $200 item, how much do I pay with delivery?",
]
for question in questions:
    fast = cag.generate_response(question, kv_cache, max_new_tokens=80).strip()  # the answer the check looks at
    checked = cag.generate_checked_response(question, kv_cache, max_new_tokens=80)
    print(f"Q: {question}\nFast answer: {fast}\n   confidence: {checked.confidence:.2f}")
    if checked.rethought:
        print(f"After thinking: {checked.answer}")
    print()

Q: How long is the TV's warranty?
Fast answer: The Vision 55 TV comes with a 2-year warranty.
   confidence: 0.85

Q: Which product has the longest warranty?
Fast answer: The FrostFree refrigerator has the longest warranty, with a 5-year warranty on the compressor.
   confidence: 0.86

Q: Does the washer come with a warranty?
Fast answer: Yes, the CleanWave washing machine comes with a 2-year warranty.
   confidence: 0.42
After thinking: The CleanWave washing machine's warranty is not mentioned in the provided context.

Q: Which is cheaper this week, the washer or the TV?
Fast answer: This week, the CleanWave washer is cheaper at $584.50, while the Vision 55 TV is $499. The washer is discounted by 10%, making it the cheaper option.
   confidence: 0.42
After thinking: The TV costs $499, while the washer is $649 with a 10% discount, making it $584.10. The TV is cheaper this week.

Q: What do the TV and the washer cost together this week, with delivery?
Fast answer: The Vision 55 TV costs

## Conversations

By default each question stands alone. `chat()` keeps the earlier turns, so a follow-up like "how much is it?" works. The knowledge cache is shared, and each follow-up reads only what is new.

In [7]:
chat = cag.chat(kv_cache)
for question in ["Do you have a 4K TV?", "How much is it?", "And how long is its warranty?"]:
    print(f"Q: {question}\nA: {chat.ask(question, max_new_tokens=60)}\n")

Q: Do you have a 4K TV?
A: Yes, we have the Vision 55, a 55-inch 4K TV with HDR10 and smart apps, priced at $499 with a 2-year warranty.

Q: How much is it?
A: The Vision 55 is $499 with a 2-year warranty.

Q: And how long is its warranty?
A: The Vision 55 comes with a 2-year warranty.



## Update the knowledge

When a document changes, update the cache instead of building it again. mantiq re-reads the knowledge only from the first change onward, so changing the offer at the end costs a fraction of reading everything.

In [8]:
documents["offers.md"] = "This week: the Vision 55 TV is $399 instead of $499."
kv_cache, seconds = cag.update_kvcache(kv_cache, documents)
print(f"Updated in {seconds:.2f} s (reading everything took {read_seconds:.2f} s)\n")

question = "Is there a deal on TVs this week?"
print(f"Q: {question}\nA: {cag.generate_response(question, kv_cache, max_new_tokens=60).strip()}")

Updated in 0.11 s (reading everything took 0.96 s)

Q: Is there a deal on TVs this week?
A: Yes, the Vision 55 TV is on sale this week for $399 instead of the regular price of $499.


## Use your own documents

- `pip install "mantiq[loaders]"`, then `LoaderFactory.get_loader("PDF", "catalog.pdf")` reads PDFs, web pages, CSV, JSON, Markdown and more.
- `prepare_kvcache_from_loader(..., saving_path="cache")` saves the cache, and `cag.read_kv_cache("cache/kv_cache.pt")` loads it in another session.
- Citation heads are built in for Qwen3, Llama 3.1 and 3.2, Gemma 3, Mistral 7B and Phi-4-mini; `cag.calibrate_citations(kv_cache)` finds them for another model in a few minutes.

More in the [README](https://github.com/Ahmed-Ousama/mantiq#readme). Questions and ideas are welcome in the [issues](https://github.com/Ahmed-Ousama/mantiq/issues).